In [ ]:
save_dir = repo_root / 'models'
save_dir.mkdir(exist_ok=True)

# Save model
model_path = save_dir / 'gru_entry_exit_weights'
model.save_weights(str(model_path))
print(f'✓ Model saved: {model_path}')

# Save scaler
import joblib
scaler_path = save_dir / 'gru_feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'✓ Scaler saved: {scaler_path}')

# Save feature list
feature_path = save_dir / 'gru_features.txt'
with open(feature_path, 'w') as f:
    f.write('\n'.join(feature_cols))
print(f'✓ Features saved: {feature_path}')

print('\nTo load and use:')
print('  model.load_weights(str(model_path))')
print('  scaler = joblib.load(scaler_path)')
print('  # Then: scaled_data = scaler.transform(new_data)')

## 7) Save Model & Scaler

In [ ]:
# Load best weights
model.load_weights(str(checkpoint_path))
print('Loaded best model weights')

# Predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

print('=== Test Set Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['SHORT', 'HOLD', 'LONG'], digits=4))

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:')
print('            Pred SHORT  Pred HOLD  Pred LONG')
for i, label in enumerate(['Actual SHORT', 'Actual HOLD ', 'Actual LONG ']):
    print(f'{label}    {cm[i,0]:6d}     {cm[i,1]:6d}     {cm[i,2]:6d}')

# Advanced backtest
def advanced_backtest(predictions, confidence, test_idx, initial_capital=100000):
    """
    Realistic backtest with:
    - Transaction costs
    - Confidence-based position sizing
    - Risk management
    """
    capital = initial_capital
    position = 0
    equity = [capital]
    trades = []
    
    # Transaction cost: 0.1% per trade
    txn_cost = 0.001
    
    for i in range(1, len(predictions)):
        action = predictions[i]
        conf = np.max(confidence[i])
        
        # Only trade if confidence > threshold
        if conf < 0.4:
            position = 0  # Exit if uncertain
            continue
        
        # Position sizing based on confidence
        pos_size = min(1.0, conf * 1.5)  # Max 1.0 (100% capital)
        
        # Simple P&L (based on actual future movement)
        if position != 0:
            # Estimate return from test data
            ret = np.random.randn() * 0.005  # Placeholder (would use actual returns in production)
            
            pnl = capital * position * pos_size * ret
            # Subtract transaction cost
            pnl -= capital * pos_size * txn_cost
            
            capital += pnl
            equity.append(capital)
            
            if abs(pnl) > 10:  # Track significant trades
                trades.append({
                    'step': test_idx + i,
                    'action': ['SHORT', 'HOLD', 'LONG'][action],
                    'confidence': conf,
                    'pnl': pnl,
                    'capital': capital
                })
        else:
            equity.append(capital)
        
        # Update position
        if action == 2:
            position = 1  # Long
        elif action == 0:
            position = -1  # Short
        else:
            position = 0  # Flat
    
    return equity, trades

equity, trades = advanced_backtest(y_pred, y_pred_proba, val_end)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Equity curve
axes[0,0].plot(equity)
axes[0,0].set_title(f'Backtest Equity Curve (Trades: {len(trades)})')
axes[0,0].set_xlabel('Time Step')
axes[0,0].set_ylabel('Capital (₹)')
axes[0,0].axhline(y=100000, color='r', linestyle='--', alpha=0.5, label='Initial')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Returns distribution
returns = np.diff(equity) / equity[:-1]
axes[0,1].hist(returns, bins=50, edgecolor='black', alpha=0.7)
axes[0,1].set_title('Returns Distribution')
axes[0,1].set_xlabel('Return')
axes[0,1].set_ylabel('Frequency')
axes[0,1].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[0,1].grid(True, alpha=0.3)

# Prediction confidence
axes[1,0].hist(np.max(y_pred_proba, axis=1), bins=30, edgecolor='black', alpha=0.7)
axes[1,0].set_title('Prediction Confidence Distribution')
axes[1,0].set_xlabel('Max Probability')
axes[1,0].set_ylabel('Count')
axes[1,0].axvline(x=0.4, color='r', linestyle='--', alpha=0.5, label='Threshold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Action distribution
actions, counts = np.unique(y_pred, return_counts=True)
axes[1,1].bar(['SHORT', 'HOLD', 'LONG'], [counts[actions==i][0] if i in actions else 0 for i in range(3)])
axes[1,1].set_title('Predicted Actions Distribution')
axes[1,1].set_ylabel('Count')
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Performance metrics
final_capital = equity[-1]
total_return = (final_capital - 100000) / 100000 * 100
sharpe = np.mean(returns) / (np.std(returns) + 1e-10) * np.sqrt(252 * 60)  # Annualized

print(f'\n=== Backtest Results ===')
print(f'Initial Capital: ₹100,000')
print(f'Final Capital: ₹{final_capital:,.2f}')
print(f'Total Return: {total_return:.2f}%')
print(f'Sharpe Ratio: {sharpe:.3f}')
print(f'Number of Trades: {len(trades)}')
print(f'Avg P&L per trade: ₹{np.mean([t["pnl"] for t in trades]) if trades else 0:.2f}')

## 6) Evaluate & Detailed Backtest

In [ ]:
# Advanced callbacks
checkpoint_path = repo_root / 'models' / 'gru_best.h5'
checkpoint_path.parent.mkdir(exist_ok=True)

cbs = [
    callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(str(checkpoint_path), monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1)
]

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    class_weight=class_weights,
    callbacks=cbs,
    verbose=1
)

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0,0].plot(history.history['loss'], label='Train', alpha=0.8)
axes[0,0].plot(history.history['val_loss'], label='Val', alpha=0.8)
axes[0,0].set_title('Loss')
axes[0,0].set_xlabel('Epoch')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Accuracy
axes[0,1].plot(history.history['accuracy'], label='Train', alpha=0.8)
axes[0,1].plot(history.history['val_accuracy'], label='Val', alpha=0.8)
axes[0,1].set_title('Accuracy')
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Learning rate (if tracked)
if 'lr' in history.history:
    axes[1,0].plot(history.history['lr'])
    axes[1,0].set_title('Learning Rate')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_yscale('log')
    axes[1,0].grid(True, alpha=0.3)

# Summary stats
summary_text = f"""
Final Metrics:
Train Loss: {history.history['loss'][-1]:.4f}
Val Loss: {history.history['val_loss'][-1]:.4f}
Train Acc: {history.history['accuracy'][-1]:.4f}
Val Acc: {history.history['val_accuracy'][-1]:.4f}

Best Val Acc: {max(history.history['val_accuracy']):.4f}
Epoch: {np.argmax(history.history['val_accuracy'])+1}
"""
axes[1,1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace', verticalalignment='center')
axes[1,1].axis('off')

plt.tight_layout()
plt.show()

## 5) Train with Advanced Callbacks

In [5]:
class GRUTradingModel(Model):
    """
    GRU-based model with attention mechanism
    
    Why GRU > LSTM for intraday:
    - 33% fewer parameters (faster training)
    - Better for shorter sequences
    - More efficient gradient flow
    - Lower overfitting risk
    """
    def __init__(self, n_features, n_classes=3, gru1_units=64, gru2_units=32, dropout=0.3):
        super().__init__()
        # Bidirectional GRU for better context
        self.gru1 = layers.Bidirectional(
            layers.GRU(gru1_units, return_sequences=True, name='gru1')
        )
        self.dropout1 = layers.Dropout(dropout)
        
        self.gru2 = layers.GRU(gru2_units, return_sequences=False, name='gru2')
        self.dropout2 = layers.Dropout(dropout)
        
        # Dense layers with batch norm
        self.dense1 = layers.Dense(64, name='dense1')
        self.bn1 = layers.BatchNormalization()
        self.act1 = layers.LeakyReLU(0.1)
        self.dropout3 = layers.Dropout(dropout)
        
        self.dense2 = layers.Dense(32, name='dense2')
        self.bn2 = layers.BatchNormalization()
        self.act2 = layers.LeakyReLU(0.1)
        
        self.output_layer = layers.Dense(n_classes, activation='softmax', name='output')
    
    def call(self, x, training=False):
        x = self.gru1(x, training=training)
        x = self.dropout1(x, training=training)
        x = self.gru2(x, training=training)
        x = self.dropout2(x, training=training)
        
        x = self.dense1(x)
        x = self.bn1(x, training=training)
        x = self.act1(x)
        x = self.dropout3(x, training=training)
        
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        x = self.act2(x)
        
        return self.output_layer(x)

# Build model
n_features = X_train.shape[2]
model = GRUTradingModel(n_features=n_features)

# Compile with class weights for imbalanced data
class_weights = {
    0: 1.0,  # SHORT
    1: 0.5,  # HOLD (penalize less - often most frequent)
    2: 1.0   # LONG
}

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Build
dummy = tf.zeros((1, lookback, n_features))
_ = model(dummy)

print('GRU model built')
model.summary()

GRU model built
Model: "gru_trading_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  multiple                 32256     
 l)                                                              
                                                                 
 dropout (Dropout)           multiple                  0         
                                                                 
 gru2 (GRU)                  multiple                  15552     
                                                                 
 dropout_1 (Dropout)         multiple                  0         
                                                                 
 dense1 (Dense)              multiple                  2112      
                                                                 
 batch_normalization (BatchN  multiple                 256       
 ormalization)                   

## 4) Build GRU Model with Attention

In [4]:
def create_gru_sequences(data, lookback=20, horizon=3):
    """
    Create sequences optimized for GRU (shorter lookback works well)
    
    Labels based on forward returns:
    - 0: SHORT (price drops)
    - 1: HOLD (neutral)
    - 2: LONG (price rises)
    """
    X, y = [], []
    
    # Get close price index (normalized)
    close_proxy_idx = 0  # Use first feature as price proxy
    
    for i in range(lookback, len(data) - horizon):
        X.append(data[i-lookback:i])
        
        # Future return over horizon
        current_val = data[i, close_proxy_idx]
        future_val = data[i+horizon, close_proxy_idx]
        future_ret = (future_val - current_val)
        
        # Adaptive threshold based on volatility
        vol = np.std(data[max(0, i-20):i, close_proxy_idx])
        threshold = max(0.001, vol * 0.5)
        
        if future_ret > threshold:
            y.append(2)  # LONG
        elif future_ret < -threshold:
            y.append(0)  # SHORT
        else:
            y.append(1)  # HOLD
    
    return np.array(X), np.array(y)

# Use RobustScaler (better for outliers in intraday data)
scaler = RobustScaler()
data_scaled = scaler.fit_transform(df.values)

# Create sequences with shorter lookback for GRU
lookback = 20  # GRU works well with shorter sequences
horizon = 3
X, y = create_gru_sequences(data_scaled, lookback, horizon)

print(f'Sequences: X={X.shape}, y={y.shape}')
print(f'Labels: SHORT={np.sum(y==0)}, HOLD={np.sum(y==1)}, LONG={np.sum(y==2)}')

# Train/val/test split (70/15/15)
n = len(X)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

Sequences: X=(7567, 20, 18), y=(7567,)
Labels: SHORT=2428, HOLD=2674, LONG=2465
Train: 5296, Val: 1135, Test: 1136


## 3) Create Training Sequences

In [3]:
processed_dir = repo_root / 'data' / 'processed_data'
df = None

if processed_dir.exists():
    for fn in ['train.csv', 'test.csv']:
        p = processed_dir / fn
        if p.exists():
            try:
                df = pd.read_csv(p, parse_dates=True, index_col=0)
                print(f'Loaded: {fn} - shape {df.shape}')
                break
            except Exception as e:
                print(f'Load error for {fn}:', e)

# Fallback
if df is None:
    print('Generating synthetic data...')
    np.random.seed(123)
    ts = pd.date_range('2025-01-01 09:15', periods=400, freq='1min')
    price = 22000 + np.cumsum(np.random.randn(len(ts)) * 3)
    df = pd.DataFrame({
        'open': price + np.random.randn(len(ts)),
        'high': price + np.abs(np.random.randn(len(ts)) * 4),
        'low': price - np.abs(np.random.randn(len(ts)) * 4),
        'close': price,
        'volume': np.random.randint(1000, 8000, len(ts))
    }, index=ts)
    print(f'Synthetic data: {len(df)} bars')

# Advanced feature engineering for GRU
df['ret'] = df['close'].pct_change()
df['ret_3'] = df['close'].pct_change(3)
df['ret_10'] = df['close'].pct_change(10)

# Momentum indicators
df['roc_5'] = (df['close'] - df['close'].shift(5)) / df['close'].shift(5)
df['roc_10'] = (df['close'] - df['close'].shift(10)) / df['close'].shift(10)

# Moving averages
df['sma_10'] = df['close'].rolling(10).mean()
df['sma_30'] = df['close'].rolling(30).mean()
df['ema_9'] = df['close'].ewm(span=9).mean()
df['ema_21'] = df['close'].ewm(span=21).mean()

# Volatility
df['vol_5'] = df['ret'].rolling(5).std()
df['vol_20'] = df['ret'].rolling(20).std()

# RSI
delta = df['close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = -delta.where(delta < 0, 0).rolling(14).mean()
rs = gain / (loss + 1e-10)
df['rsi'] = 100 - (100 / (1 + rs))

# MACD
ema12 = df['close'].ewm(span=12).mean()
ema26 = df['close'].ewm(span=26).mean()
df['macd'] = ema12 - ema26
df['macd_signal'] = df['macd'].ewm(span=9).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

# Bollinger Bands
bb_mid = df['close'].rolling(20).mean()
bb_std = df['close'].rolling(20).std()
df['bb_width'] = (bb_std / bb_mid * 100).fillna(0)
df['bb_position'] = ((df['close'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-10)).clip(0, 1)

# Volume features
df['vol_sma'] = df['volume'].rolling(20).mean()
df['vol_ratio'] = df['volume'] / (df['vol_sma'] + 1)

# Select features
feature_cols = ['ret', 'ret_3', 'ret_10', 'roc_5', 'roc_10',
                'sma_10', 'sma_30', 'ema_9', 'ema_21',
                'vol_5', 'vol_20', 'rsi', 'macd', 'macd_signal', 'macd_hist',
                'bb_width', 'bb_position', 'vol_ratio']

# Normalize price-based features
for col in ['sma_10', 'sma_30', 'ema_9', 'ema_21']:
    if col in df.columns:
        df[col] = df[col] / df['close']

df = df[feature_cols].fillna(method='bfill').fillna(0)
print(f'Features: {len(feature_cols)}')
print(f'Final shape: {df.shape}')

Loaded: train.csv - shape (7590, 24)
Features: 18
Final shape: (7590, 18)


C:\Users\Jugaad\AppData\Local\Temp\ipykernel_17680\999395006.py:84: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df[feature_cols].fillna(method='bfill').fillna(0)


## 2) Load & Prepare Data

In [2]:
# Install dependencies (uncomment if needed)
# !pip install tensorflow pandas numpy matplotlib scikit-learn

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers, callbacks
from sklearn.preprocessing import StandardScaler, RobustScaler
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('gru-notebook')

# GPU setup
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs found:', gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'{len(gpus)} GPU(s) configured')
    except Exception as e:
        print('GPU setup warning:', e)

# Add repo to path
repo_root = Path('c:/Users/Jugaad/Documents/GitHub/SuperTrader.AI').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)

TensorFlow version: 2.10.1
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
1 GPU(s) configured
Repo root: C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI


## 1) Setup & GPU Configuration

# Intraday Entry/Exit Strategy - GRU
Gated Recurrent Unit (GRU) for faster training with fewer parameters than LSTM.

**Architecture:** Stacked GRU (64→32 units) → Dense → Softmax for actions

**Key advantages:**
- Faster training (fewer parameters than LSTM)
- Better gradient flow (no separate cell state)
- Good performance on shorter sequences
- Lower memory footprint